##Gold Layer for Analysis of AF, SCB and UKÄ datasets

## Build `dim_occupation`

This dimension standardizes occupation data across JobTech and SCB by using SSYK as the common classification.

### Purpose
`dim_occupation` provides a conformed occupation dimension that can be shared across:
- `fact_job_postings`
- `fact_wages`
- `fact_employment`

### Source
The dimension is built from the occupation mapping table created during the harmonization step:
- `silver_jobtech.occupation_group_to_ssyk_mapping`

### Design choice
The dimension is SSYK-centered and preserves the original JobTech occupation group label.

In [0]:
CREATE SCHEMA IF NOT EXISTS gold_analysis

In [0]:
DESCRIBE silver_jobtech.occupation_group_to_ssyk_mapping;

## Create `dim_occupation`

This step creates the Gold occupation dimension from the occupation mapping table.

### Structure
The dimension includes:
- a surrogate key: `occupation_id`
- the standardized SSYK code and name
- the original JobTech occupation group label


In [0]:
CREATE OR REPLACE TABLE gold_analysis.dim_occupation AS
SELECT
  ROW_NUMBER() OVER (
    ORDER BY
      -- If occupation_code is null, replace it with 'ZZZZ'
      coalesce(occupation_code, 'ZZZZ'),
      occupation_group_label
  ) AS occupation_id,
  occupation_code AS ssyk_code,
  occupation_name AS ssyk_name,
  occupation_group_label AS jobtech_occupation_group_label
FROM silver_jobtech.occupation_group_to_ssyk_mapping;

In [0]:
SELECT *
FROM gold_analysis.dim_occupation
ORDER BY occupation_id
LIMIT 20;

## Validate `dim_occupation`

This step validates the Gold occupation dimension after creation.

### What is being checked
- total number of rows in the dimension
- consistency with the occupation mapping source

### Why this matters
This confirms that the conformed occupation dimension was created successfully and is ready to be used by Gold fact tables.

In [0]:
SELECT COUNT(*) AS dim_occupation_rows
FROM gold_analysis.dim_occupation;

In [0]:
SELECT COUNT(*) AS mapping_rows
FROM silver_jobtech.occupation_group_to_ssyk_mapping;

In [0]:
CREATE OR REPLACE TABLE gold_analysis.dim_date (
  date_id DATE,
  full_date DATE,
  year INT,
  month INT,
  quarter INT,
  week INT,
  day_of_week STRING
);

Dim_date uses a natural key where date_id is the actual date. 

In [0]:
INSERT INTO gold_analysis.dim_date 
WITH date_range AS (
  SELECT
    explode(
      SEQUENCE(
        (SELECT MIN(publication_date) FROM ...),
        DATE_ADD(CURRENT_DATE, 365),  -- 1 year ahead
        INTERVAL 1 DAY
      )
    ) AS full_date
),
date_details AS (
  SELECT
    full_date AS date_id,
    YEAR(full_date) AS year,
    MONTH(full_date) AS month,
    QUARTER(full_date) AS quarter,
    WEEKOFYEAR(full_date) AS week,
    DATE_FORMAT(full_date, 'EEEE') AS day_of_week
  FROM date_range
)
SELECT * FROM date_details;


In [0]:
CREATE OR REPLACE TABLE gold_analysis.dim_gender (
  gender_id INT,
  gender_name STRING,
  is_actual_gender BOOLEAN
);

In [0]:
CREATE OR REPLACE TABLE gold_analysis.dim_location (
  location_ID INT,
  municipality_name STRING,
  municipality_code STRING,
  region_name STRING
);

In [0]:
TRUNCATE TABLE gold_analysis.dim_gender;

INSERT INTO gold_analysis.dim_gender
SELECT 
  CASE 
    WHEN gender_name = 'men' THEN 1
    WHEN gender_name = 'women' THEN 2
    WHEN LOWER(gender_name) = 'total' THEN 3
    ELSE 99
  END AS gender_id,
  gender_name,
  CASE 
    WHEN LOWER(gender_name) = 'total' THEN FALSE
    ELSE TRUE
  END AS is_actual_gender
FROM (
  SELECT DISTINCT gender AS gender_name FROM silver_education.university_uka
  UNION
  SELECT DISTINCT gender FROM silver_education.yh_region_subject
  UNION
  SELECT DISTINCT gender FROM labour_market_platform.silver_work_scb.aku_population
  UNION
  SELECT DISTINCT gender FROM labour_market_platform.silver_work_scb.aku_employment
  UNION
  SELECT DISTINCT gender FROM labour_market_platform.silver_work_scb.aku_unemployment
  UNION
  SELECT DISTINCT gender FROM labour_market_platform.silver_work_scb.wages
)
ORDER BY gender_id;

In [0]:
SELECT * FROM gold_analysis.dim_gender;

In [0]:
SELECT DISTINCT region from silver_work_scb.aku_population

In [0]:
SELECT DISTINCT region_name FROM silver_education.yh_region_subject

## Build `fact_job_postings`

### Grain
One row per job posting (`job_id`) from the cleaned and deduplicated JobTech Silver table.

### Planned dimensions
- `occupation_id`
- `date_id`
- `location_id`

### Planned measures / attributes
- `number_of_vacancies`
- `scope_of_work_min`
- `scope_of_work_max`
- `application_deadline`
- `employment_type_label`
- `duration_label`
- `working_hours_type_label`

### Source
`silver_jobtech.job_postings_silver_clean`

At this step, we first validate the join to `dim_occupation` before building the final fact table.

In [0]:
SELECT
  COUNT (*) AS total_job_postings,
  SUM(CASE WHEN d.occupation_id IS NOT NULL THEN 1 ELSE 0 END) AS matched_to_dim_occupation,
  SUM(CASE WHEN d.occupation_id IS NULL THEN 1 ELSE 0 END) AS unmatched_to_dim_occupation,
  ROUND(
        100.0 * SUM(CASE WHEN d.occupation_id IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS occupation_match_pct
FROM silver_jobtech.job_postings_silver_clean j
LEFT JOIN gold_analysis.dim_occupation d
ON j.occupation_code = d.occupation_code
SELECT
  COUNT(*) AS total_job_postings,